# [9.2] Chain-of-Thought Faithfulness - Solutions

This notebook runs the reference implementations and reproduces the visible checks. The source implementations live in `solutions.py`; the cells below keep the same conceptual arc as the exercise notebook and show the expected outputs.

<details>
<summary>Expected output</summary>

All visible tests should pass, the CPU smoke contract should match the toy expected values, and the committed CUDA signature table should match the report.
</details>

<details>
<summary>Help - what should I focus on?</summary>

The main lesson is the evidence ladder: probe readout, readout patch, text-only baseline, label-shuffle control, condition split, and narrow claim boundary.
</details>


In [1]:
import json
import sys
from pathlib import Path

chapter = "chapter9_alignment_interpretability"
section = "part2_cot_faithfulness"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_cot_faithfulness.solutions as solutions
import part2_cot_faithfulness.tests as tests
import part2_cot_faithfulness.utils as utils


## Visible Unit Tests

<details>
<summary>Expected output</summary>

Each test prints `All tests in ... passed!`.
</details>

<details>
<summary>Help - why so many tiny tests?</summary>

ARENA notebooks catch mistakes immediately after the function where they happen. These tests make axis errors, missing finite checks, and baseline confusion visible before the report stage.
</details>


In [2]:
tests.test_prediction_accuracy_checks_top1_predictions(solutions.prediction_accuracy)
tests.test_pre_final_answer_probe_report_predicts_hidden_answer(
    solutions.pre_final_answer_probe_report,
)
tests.test_hidden_answer_patching_report_flags_answer_flip(
    solutions.hidden_answer_patching_report,
)
tests.test_cot_text_baseline_report_keeps_recall_gap(solutions.cot_text_baseline_report)
tests.test_feature_detector_report_scores_thresholded_predictions(
    solutions.feature_detector_report,
)
tests.test_cot_condition_comparison_report_tracks_gaps(
    solutions.cot_condition_comparison_report,
)


All tests in `test_prediction_accuracy_checks_top1_predictions` passed!
All tests in `test_pre_final_answer_probe_report_predicts_hidden_answer` passed!
All tests in `test_hidden_answer_patching_report_flags_answer_flip` passed!
All tests in `test_cot_text_baseline_report_keeps_recall_gap` passed!
All tests in `test_feature_detector_report_scores_thresholded_predictions` passed!
All tests in `test_cot_condition_comparison_report_tracks_gaps` passed!


## Notebook Contract

<details>
<summary>Expected output</summary>

The probe should have hidden-answer accuracy `1.0`, final-answer agreement `2/3`; the patch should flip `0 -> 1`; the detector should beat the text-only baseline.
</details>

<details>
<summary>Help - interpreting the toy contract</summary>

The toy contract proves the local functions behave as intended. It is deliberately not the GT-3 evidence path.
</details>


In [3]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["probe"]["hidden_answer_accuracy"] == 1.0
assert abs(contract["probe"]["final_answer_agreement"] - (2 / 3)) < 1e-6
assert contract["patching"]["changed_output"]
assert contract["text_baseline"]["detector_recall"] == 1.0
assert contract["text_baseline"]["text_only_recall"] == 0.5
assert contract["feature_detector"]["feature_accuracy"] == 1.0
assert contract["feature_detector"]["baseline_accuracy"] == 0.75
assert abs(contract["condition_comparison"]["biased_gap"] - (1 / 3)) < 1e-6
utils.print_report("CPU smoke contract", contract)


CPU smoke contract
  probe                : {'hidden_answer_accuracy': 1.0, 'final_answer_agreement': 0.6666666865348816, 'predicts_hidden_answer': True}
  patching             : {'original_answer': 0, 'patched_answer': 1, 'changed_output': True}
  text_baseline        : {'detector_recall': 1.0, 'text_only_recall': 0.5, 'text_only_misses_cases': True}
  feature_detector     : {'feature_accuracy': 1.0, 'baseline_accuracy': 0.75, 'improves_detection': True}
  condition_comparison : {'condition_accuracies': {'no_cot': 0.6666666865348816, 'faithful_cot': 1.0, 'biased_cot': 0.3333333432674408, 'posthoc': 0.6666666865348816}, 'biased_gap': 0.3333333432674408, 'posthoc_gap': 0.3333333134651184}


## Signature Result

<img src="../../instructions/assets/cot_faithfulness_signature_result.svg" width="860">

<details>
<summary>Interpreting the signature result</summary>

The report shows hidden-answer information is readable and that an LM-head readout flip is possible after moving the hidden vector. The result is real CUDA evidence for the scoped Pythia preflight, not a broad claim about generated reasoning.
</details>


In [4]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]

assert report["accepted"]
assert report["tests_passed"]
assert gpu["cuda_available"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "EleutherAI/pythia-70m-deduped"
assert gpu["hf_revision"] == "e93a9faa9c77e5d09219f6c868bfc7a1bd65593c"
assert gpu["hidden_answer_accuracy"] == 1.0
assert gpu["final_answer_agreement"] == 0.5
assert gpu["label_shuffled_probe_accuracy"] == 0.0
assert gpu["patching_changed_output"]
assert gpu["text_only_recall"] == 0.5
assert gpu["baseline_detector_accuracy"] == 0.75
assert gpu["train_prompt_count"] == 24
assert gpu["heldout_prompt_count"] == 8
assert gpu["hidden_state_shape"] == [8, 512]
assert gpu["within_vram_budget"]

tests.test_committed_gpu_report_uses_real_text_only_baseline(gpu)
utils.print_report(
    "Committed CUDA report",
    {
        "device": gpu["device"],
        "hidden_answer_accuracy": gpu["hidden_answer_accuracy"],
        "final_answer_agreement": gpu["final_answer_agreement"],
        "model_answer_accuracy": gpu["model_answer_accuracy"],
        "label_shuffled_probe_accuracy": gpu["label_shuffled_probe_accuracy"],
        "patching_changed_output": gpu["patching_changed_output"],
        "detector_vs_text_recall": f'{gpu["detector_recall"]} vs {gpu["text_only_recall"]}',
        "condition_accuracies": gpu["condition_accuracies"],
        "posthoc_gap": gpu["posthoc_gap"],
        "peak_vram_gb": gpu["peak_vram_gb"],
    },
)


All tests in `test_committed_gpu_report_uses_real_text_only_baseline` passed!
Committed CUDA report
  device                        : NVIDIA GeForce RTX 5090 Laptop GPU
  hidden_answer_accuracy        : 1.0
  final_answer_agreement        : 0.5
  model_answer_accuracy         : 0.625
  label_shuffled_probe_accuracy : 0.0
  patching_changed_output       : True
  detector_vs_text_recall       : 1.0 vs 0.5
  condition_accuracies          : {'biased_cot': 0.5, 'faithful_cot': 0.5, 'no_cot': 0.5, 'posthoc': 1.0}
  posthoc_gap                   : -0.5
  peak_vram_gb                  : 0.33783864974975586


## Limitations

- The real path uses safe synthetic A/B prompts and answer-token logits only.
- The hidden-vector patch is an LM-head readout test, not a full forward-pass activation patch.
- The held-out set is small and Pythia-70M-specific.
- The post-hoc condition is stronger than the faithful condition in this tiny report, so condition metrics should be read as diagnostics rather than broad support.

## Further Research

Layer sweeps, stronger text baselines, random-shuffle distributions, nonlinear probes, and full hook-based patching are the natural next experiments.
